In [2]:
import pandas as pd

# Load the training dataset
print("Loading data...")
df = pd.read_csv("data/fraudTrain.csv")

# Print the shape (number of rows and columns)
print(f"Dataset Shape: {df.shape}")

# Show the first 5 rows to see what the data looks like
df.head()

Loading data...
Dataset Shape: (1296675, 23)


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  str    
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  str    
 4   category               1296675 non-null  str    
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  str    
 7   last                   1296675 non-null  str    
 8   gender                 1296675 non-null  str    
 9   street                 1296675 non-null  str    
 10  city                   1296675 non-null  str    
 11  state                  1296675 non-null  str    
 12  zip                    1296675 non-null  int64  
 13  lat                    1296675 non-null  float64
 14  long                   129667

In [4]:
# 1. Drop useless columns
df = df.drop(columns=['Unnamed: 0'])

# 2. Create 'card_bin' by converting the credit card number to a string and taking the first 6 characters
print("Extracting Card BINs...")
df['card_bin'] = df['cc_num'].astype(str).str[:6]

# 3. Let's look at the new data frame to confirm it worked!
display(df[['cc_num', 'card_bin', 'amt', 'is_fraud']].head())

Extracting Card BINs...


,cc_num,card_bin,amt,is_fraud
0,2703186189652095,270318,4.97,0
1,630423337322,630423,107.23,0
2,38859492057661,388594,220.11,0
3,3534093764340240,353409,45.00,0
4,375534208663984,375534,41.96,0


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Define Features (X) and Target (y)
# We are using simple numerical columns for our baseline test
features = ['amt', 'city_pop', 'zip']
X = df[features]
y = df['is_fraud']

# 2. Split the Data
# We hide 20% of the data from the AI to test it later (like a final exam)
print("Splitting data into Training and Testing sets...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize and Train the Model
print("Training the Baseline Model (Logistic Regression)...")
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 4. Grade the AI's Final Exam
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Model Training Complete!")
print(f"Baseline Accuracy: {accuracy * 100:.2f}%")

Splitting data into Training and Testing sets...
Training the Baseline Model (Logistic Regression)...
Model Training Complete!
Baseline Accuracy: 99.36%


In [6]:
from sklearn.metrics import classification_report, confusion_matrix

print("--- AI Truth Serum ---")
print("Classification Report:\n")
# This will show us Precision, Recall, and F1-Score for both classes (0 and 1)
print(classification_report(y_test, predictions))

print("\nConfusion Matrix:\n")
# This shows exactly how many it got right and wrong
print(confusion_matrix(y_test, predictions))

--- AI Truth Serum ---
Classification Report:

              precision    recall  f1-score   support

           0       0.99      1.00      1.00    257815
           1       0.00      0.00      0.00      1520

    accuracy                           0.99    259335
   macro avg       0.50      0.50      0.50    259335
weighted avg       0.99      0.99      0.99    259335


Confusion Matrix:

[[257668    147]
 [  1520      0]]


In [7]:
# Train the model but force it to care about the rare class!
print("Training Balanced Logistic Regression...")
balanced_model = LogisticRegression(max_iter=1000, class_weight='balanced')
balanced_model.fit(X_train, y_train)

# Grade it again
balanced_predictions = balanced_model.predict(X_test)

print("\n--- NEW AI Truth Serum ---")
print(classification_report(y_test, balanced_predictions))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, balanced_predictions))

Training Balanced Logistic Regression...



--- NEW AI Truth Serum ---
              precision    recall  f1-score   support

           0       1.00      0.95      0.98    257815
           1       0.09      0.78      0.16      1520

    accuracy                           0.95    259335
   macro avg       0.54      0.86      0.57    259335
weighted avg       0.99      0.95      0.97    259335


Confusion Matrix:

[[245637  12178]
 [   340   1180]]


In [8]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

print("Training the Heavy Artillery (XGBoost)...")

# Calculate the exact imbalance ratio for XGBoost
neg_class_count = len(y_train[y_train == 0])
pos_class_count = len(y_train[y_train == 1])
scale_weight = neg_class_count / pos_class_count

# Initialize and train the model
xgb_model = XGBClassifier(scale_pos_weight=scale_weight, random_state=42)
xgb_model.fit(X_train, y_train)

# Grade the final exam
xgb_predictions = xgb_model.predict(X_test)

print("\n--- XGBoost Truth Serum ---")
print(classification_report(y_test, xgb_predictions))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, xgb_predictions))

Training the Heavy Artillery (XGBoost)...

--- XGBoost Truth Serum ---
              precision    recall  f1-score   support

           0       1.00      0.93      0.97    257815
           1       0.07      0.85      0.13      1520

    accuracy                           0.93    259335
   macro avg       0.53      0.89      0.55    259335
weighted avg       0.99      0.93      0.96    259335


Confusion Matrix:

[[240801  17014]
 [   222   1298]]


In [9]:
import joblib

# Save the trained XGBoost model to a file
model_filename = 'xgboost_fraud_model.joblib'
joblib.dump(xgb_model, model_filename)

print(f"Success! AI brain saved as: {model_filename}")

Success! AI brain saved as: xgboost_fraud_model.joblib
